# Autonomous Driving Danger Detection System
## Multi-Camera Perception | nuScenes Dataset

| Module | Model | Paper | Year |
|---|---|---|---|
| Object Detection | **YOLO11x** | Ultralytics | 2024 |
| Depth Estimation | **Depth Anything V2 Metric Outdoor** | Yang et al., NeurIPS | 2024 |
| Multi-Object Tracking | **StrongSORT** (via BoxMOT) | Du et al., IEEE TMM | 2023 |
| Optical Flow | **RAFT** (fallback: Farneback) | Teed & Deng, ECCV | 2020 |
| Panorama Stitching | **OpenCV SIFT + Homography** | — | — |
| Risk Scoring | **TTC + Multi-factor** | — | — |
| BEV | **nuScenes HD Map + Model outputs** | nuScenes | — |


In [1]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import subprocess, sys, os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print('ERR:', r.stderr[-200:])
    return r.returncode == 0

run('pip install -q "numpy<2.0"')
run('pip install -q nuscenes-devkit==1.2.0')
run('pip install -q ultralytics==8.3.40')
run('pip install -q pyquaternion==0.9.9')
run('pip install -q "imageio[ffmpeg]"')
run('pip install -q tqdm')
run('pip install -q "transformers>=4.50.0" "regex>=2025.10.22"')
run('pip install -q boxmot')

if not os.path.exists('/content/RAFT'):
    run('git clone -q https://github.com/princeton-vl/RAFT.git /content/RAFT')

print('All installed. Now: Runtime → Restart Runtime, then run from Cell 2.')


All installed. Now: Runtime → Restart Runtime, then run from Cell 2.


In [1]:
# ── CELL 2: Imports ──────────────────────────────────────────────────────────
import sys, os, math, warnings, tempfile
from collections import defaultdict, deque

import numpy as np
import cv2
import torch
import imageio
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm import tqdm

from ultralytics import YOLO
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

sys.path.insert(0, '/content/RAFT/core')
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  torch: {torch.__version__}')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda  |  torch: 2.11.0+cu128


In [2]:
# ── CELL 3: Load nuScenes + Map from Google Drive ────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

NUSCENES_ROOT = '/content/drive/MyDrive/datasets/nuscenes'
os.makedirs(NUSCENES_ROOT, exist_ok=True)

if not os.path.exists(f'{NUSCENES_ROOT}/v1.0-mini'):
    print('Extracting nuScenes mini...')
    os.system(f'tar -xf "{NUSCENES_ROOT}/v1.0-mini.tgz" -C {NUSCENES_ROOT}')

if not os.path.exists(f'{NUSCENES_ROOT}/maps/expansion'):
    print('Extracting map expansion...')
    os.makedirs(f'{NUSCENES_ROOT}/maps/expansion', exist_ok=True)
    os.system(f'unzip -q "{NUSCENES_ROOT}/nuScenes-map-expansion-v1.3.zip" -d /tmp/map_ext')
    os.system(f'find /tmp/map_ext -name "*.json" -exec cp {{}} {NUSCENES_ROOT}/maps/expansion/ ;')
    os.system(f'find /tmp/map_ext -name "*.png"  -exec cp {{}} {NUSCENES_ROOT}/maps/ ;')
    print('Maps:', os.listdir(f'{NUSCENES_ROOT}/maps/expansion'))

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)
print(f'nuScenes loaded: {len(nusc.scene)} scenes')

MAP_LOCS = ['singapore-onenorth','singapore-queenstown',
            'singapore-hollandvillage','boston-seaport']
nusc_map = selected_scene = None
for scene in nusc.scene:
    log = nusc.get('log', scene['log_token'])
    loc = log['location']
    if loc in MAP_LOCS:
        try:
            nusc_map = NuScenesMap(dataroot=NUSCENES_ROOT, map_name=loc)
            selected_scene = scene
            print(f'Scene: {scene["name"]}  |  {loc}')
            break
        except Exception as e:
            print(f'Map fail {loc}: {e}')

assert selected_scene is not None, 'No scene with map found.'

samples = []
tok = selected_scene['first_sample_token']
while tok:
    s = nusc.get('sample', tok)
    samples.append(s)
    tok = s['next']
print(f'Frames: {len(samples)}')

CAM_NAMES  = ['CAM_FRONT','CAM_FRONT_RIGHT','CAM_FRONT_LEFT',
              'CAM_BACK', 'CAM_BACK_LEFT',  'CAM_BACK_RIGHT']
FRONT_CAMS = ['CAM_FRONT_LEFT','CAM_FRONT','CAM_FRONT_RIGHT']
BACK_CAMS  = ['CAM_BACK_RIGHT','CAM_BACK','CAM_BACK_LEFT']


Mounted at /content/drive
nuScenes loaded: 10 scenes
Scene: scene-0061  |  singapore-onenorth
Frames: 39


In [3]:
# ── CELL 4: Load YOLO11x + proper COCO <-> nuScenes class mapping ────────────
yolo = YOLO('yolo11x.pt')
yolo.to(DEVICE)

# === FIX 1: Complete COCO-to-nuScenes class mapping ===
# YOLO trained on COCO; nuScenes uses different taxonomy.
# Without proper mapping, 'adult' never matches 'person' -> inflated FN.
DETECT_CLASSES = {
    0:'person', 1:'bicycle', 2:'car', 3:'motorcycle',
    5:'bus', 7:'truck', 15:'cat', 16:'dog', 17:'horse',
    18:'sheep', 19:'cow',
}

# nuScenes category -> COCO class name (for evaluation matching)
NUSCENES_TO_COCO = {
    'human.pedestrian.adult':       'person',
    'human.pedestrian.child':       'person',
    'human.pedestrian.construction_worker': 'person',
    'human.pedestrian.police_officer':      'person',
    'human.pedestrian.personal_mobility':   'person',
    'human.pedestrian.stroller':    'person',
    'human.pedestrian.wheelchair':  'person',
    'vehicle.car':                  'car',
    'vehicle.truck':                'truck',
    'vehicle.bus.bendy':            'bus',
    'vehicle.bus.rigid':            'bus',
    'vehicle.bicycle':              'bicycle',
    'vehicle.motorcycle':           'motorcycle',
    'animal':                       'dog',
    'vehicle.construction':         'truck',
    'vehicle.trailer':              'truck',
}

print('YOLO11x loaded with proper class mapping.')


100%|██████████| 109M/109M [00:00<00:00, 133MB/s] 


YOLO11x loaded with proper class mapping.


In [4]:
# ── CELL 5: Depth Anything V2 Metric Outdoor (NeurIPS 2024) ──────────────────
print('Loading Depth Anything V2...')
da2_processor = AutoImageProcessor.from_pretrained(
    'depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf')
da2_model = AutoModelForDepthEstimation.from_pretrained(
    'depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf').to(DEVICE)
da2_model.eval()
print('Depth Anything V2 loaded.')


Loading Depth Anything V2...


preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

Depth Anything V2 loaded.


In [7]:
# ── CELL 6: StrongSORT tracker — all 6 cameras (BoxMOT, Du et al. IEEE TMM 2023)
# Fix: StrongSort renamed to Boxmot with tracker='strongsort' in newer boxmot versions
from boxmot import Boxmot
from pathlib import Path

trackers = {}
for cam in CAM_NAMES:
    trackers[cam] = Boxmot(
        tracker='strongsort',
        reid=Path('osnet_x0_25_msmt17.pt'),
    )
print('StrongSORT trackers initialized for all 6 cameras.')

StrongSORT trackers initialized for all 6 cameras.


In [8]:
# ── CELL 7: RAFT Optical Flow (Teed & Deng, ECCV 2020) ───────────────────────
import sys
sys.path.insert(0, '/content/RAFT/core')

try:
    from raft import RAFT
    from utils.utils import InputPadder
    import argparse
    args = argparse.Namespace(small=False, mixed_precision=False, alternate_corr=False)
    if not os.path.exists('/content/models/raft-things.pth'):
        os.system('bash /content/RAFT/download_models.sh')
    raft_model = RAFT(args)
    raft_model = torch.nn.DataParallel(raft_model)
    raft_model.load_state_dict(torch.load('/content/models/raft-things.pth', map_location=DEVICE))
    raft_model = raft_model.module.to(DEVICE).eval()
    USE_RAFT = True
    print('RAFT loaded.')
except Exception as e:
    print(f'RAFT load failed: {e} -- Falling back to Farneback.')
    USE_RAFT = False


RAFT loaded.


In [9]:
# ── CELL 8: nuScenes helper functions (with ego speed + visibility) ──────────

def get_ego_pose(sample):
    sd  = nusc.get('sample_data', sample['data']['CAM_FRONT'])
    ep  = nusc.get('ego_pose', sd['ego_pose_token'])
    x, y = ep['translation'][0], ep['translation'][1]
    q    = ep['rotation']
    yaw  = math.atan2(2*(q[0]*q[3]+q[1]*q[2]), 1-2*(q[2]**2+q[3]**2))
    return float(x), float(y), float(yaw)

def get_ego_speed(ego_hist, dt=0.5):
    """Compute ego speed (m/s) from position history. nuScenes keyframes ~0.5s apart."""
    if len(ego_hist) < 2:
        return 0.0
    x1, y1 = ego_hist[-2]
    x2, y2 = ego_hist[-1]
    return math.hypot(x2 - x1, y2 - y1) / dt

def get_cam_image(sample, cam):
    sd  = nusc.get('sample_data', sample['data'][cam])
    img = cv2.imread(os.path.join(NUSCENES_ROOT, sd['filename']))
    return img

def get_intrinsics(sample, cam):
    sd = nusc.get('sample_data', sample['data'][cam])
    cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    return np.array(cs['camera_intrinsic'], dtype=np.float64)

def get_cam2ego(sample, cam):
    sd  = nusc.get('sample_data', sample['data'][cam])
    cs  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    R   = Quaternion(cs['rotation']).rotation_matrix
    t   = np.array(cs['translation'])
    T   = np.eye(4, dtype=np.float64)
    T[:3,:3] = R; T[:3,3] = t
    return T

def get_ego2world(sample):
    """Full ego->world 4x4 transform."""
    sd = nusc.get('sample_data', sample['data']['CAM_FRONT'])
    ep = nusc.get('ego_pose', sd['ego_pose_token'])
    R  = Quaternion(ep['rotation']).rotation_matrix
    t  = np.array(ep['translation'])
    T  = np.eye(4, dtype=np.float64)
    T[:3,:3] = R; T[:3,3] = t
    return T

def is_visible_in_cam(sample, cam, world_point):
    """Check if a 3D world point projects into camera's image plane."""
    T_e2w = get_ego2world(sample)
    T_c2e = get_cam2ego(sample, cam)
    K = get_intrinsics(sample, cam)
    pt_ego = np.linalg.inv(T_e2w) @ np.append(world_point[:3], 1.0)
    pt_cam = np.linalg.inv(T_c2e) @ pt_ego
    if pt_cam[2] <= 0:
        return False, None
    px = K[0,0] * pt_cam[0] / pt_cam[2] + K[0,2]
    py = K[1,1] * pt_cam[1] / pt_cam[2] + K[1,2]
    if 0 <= px < 1600 and 0 <= py < 900:
        return True, float(pt_cam[2])
    return False, None

print('nuScenes helpers defined (with ego speed + visibility check).')


nuScenes helpers defined (with ego speed + visibility check).


In [10]:
# ── CELL 9: Depth estimation helpers (improved bbox depth) ───────────────────

def run_depth(img_bgr):
    """Run Depth Anything V2 on a BGR image -> metric depth map (meters)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = PILImage.fromarray(img_rgb)
    inputs  = da2_processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = da2_model(**inputs)
    depth = da2_processor.post_process_depth_estimation(
        outputs,
        target_sizes=[(img_bgr.shape[0], img_bgr.shape[1])]
    )[0]['predicted_depth']
    return depth.squeeze().float().cpu().numpy()

def bbox_depth(depth_map, bbox):
    """Get depth for a bounding box using center 50% crop.
    === FIX 3: Using central region avoids background/edge pixels
    that corrupt the depth estimate (especially for thin objects). ==="""
    h, w = depth_map.shape
    x1 = max(0, int(bbox[0])); y1 = max(0, int(bbox[1]))
    x2 = min(w, int(bbox[2])); y2 = min(h, int(bbox[3]))
    bw = x2 - x1; bh = y2 - y1
    # Center 50% crop
    cx1 = x1 + bw // 4; cx2 = x2 - bw // 4
    cy1 = y1 + bh // 4; cy2 = y2 - bh // 4
    patch = depth_map[cy1:cy2, cx1:cx2]
    if patch.size == 0:
        patch = depth_map[y1:y2, x1:x2]
    if patch.size == 0:
        return float('inf')
    valid = patch[(patch > 0.3) & (patch < 80)]
    if valid.size == 0:
        return float(np.median(patch))
    return float(np.median(valid))

print('Depth helpers defined (center-crop method).')


Depth helpers defined (center-crop method).


In [11]:
# ── CELL 10: Optical flow velocity estimator ─────────────────────────────────

prev_frames = {}

def get_flow(cam, img_bgr):
    """Compute optical flow between current and previous frame."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    flow = None
    if cam in prev_frames:
        if USE_RAFT:
            try:
                from utils.utils import InputPadder
                def to_tensor(g):
                    rgb = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
                    return torch.from_numpy(rgb).permute(2,0,1).float()[None].to(DEVICE)
                img1 = to_tensor(prev_frames[cam])
                img2 = to_tensor(gray)
                padder = InputPadder(img1.shape)
                img1, img2 = padder.pad(img1, img2)
                with torch.no_grad():
                    _, flow_up = raft_model(img1, img2, iters=12, test_mode=True)
                flow = flow_up[0].permute(1,2,0).cpu().numpy()
            except Exception:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_frames[cam], gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        else:
            flow = cv2.calcOpticalFlowFarneback(
                prev_frames[cam], gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    prev_frames[cam] = gray
    return flow

def bbox_flow(flow, bbox):
    """Get median flow vector within a bounding box."""
    if flow is None: return 0.0, 0.0, 0.0, 0.0
    x1,y1,x2,y2 = int(bbox[0]),int(bbox[1]),int(bbox[2]),int(bbox[3])
    h, w = flow.shape[:2]
    x1=max(0,x1); y1=max(0,y1); x2=min(w,x2); y2=min(h,y2)
    patch = flow[y1:y2, x1:x2]
    if patch.size == 0: return 0.0, 0.0, 0.0, 0.0
    vx = float(np.median(patch[...,0]))
    vy = float(np.median(patch[...,1]))
    return vx, vy, math.hypot(vx,vy), math.atan2(vy,vx)

print('Optical flow helpers defined.')


Optical flow helpers defined.


In [12]:
# ── CELL 11: Risk scoring with ego speed, TTC, camera priority ──────────────
# === FIX 4: Multi-factor risk considering ego velocity + relative motion ===

CLASS_W = {
    'person':1.5, 'bicycle':1.3, 'motorcycle':1.3,
    'car':1.0, 'truck':1.1, 'bus':1.1,
    'horse':1.0, 'cow':0.8, 'dog':0.9,
}

RISK_COLOR_BGR = {
    'HIGH':   (50,  50,  220),
    'MEDIUM': (0,  140,  255),
    'LOW':    (0,  190,  190),
    'SAFE':   (80, 160,   80),
}

def assess_risk(depth_m, flow_speed_px=0.0, class_name='car',
                ego_speed_ms=0.0, obj_heading_rad=0.0, cam_name='CAM_FRONT'):
    """
    Comprehensive risk assessment:
    - Proximity (depth in meters)
    - Object flow speed (proxy for relative velocity)
    - Ego vehicle speed (faster = less reaction time)
    - TTC (Time To Collision)
    - Camera priority (front > rear)
    - Class weight (pedestrians more vulnerable)
    """
    cw = CLASS_W.get(class_name, 1.0)

    # Proximity score
    if   depth_m < 5:   prox = 1.0
    elif depth_m < 10:  prox = 0.85
    elif depth_m < 20:  prox = 0.5
    elif depth_m < 30:  prox = 0.25
    else:               prox = 0.1

    # Flow speed score
    flow_norm = min(1.0, flow_speed_px / 15.0)

    # Ego speed factor
    ego_factor = min(1.0, ego_speed_ms / 15.0)

    # TTC estimate
    closing_speed = ego_speed_ms + flow_speed_px * 0.3
    ttc = depth_m / max(closing_speed, 0.5)
    ttc_score = max(0.0, 1.0 - ttc / 8.0)

    # Camera priority
    cam_weight = 1.0 if 'FRONT' in cam_name else 0.7

    raw = cw * cam_weight * (0.35*prox + 0.20*flow_norm + 0.15*ego_factor + 0.30*ttc_score)
    score = min(1.0, raw)

    if   score >= 0.65 or depth_m < 5:   level = 'HIGH'
    elif score >= 0.40 or depth_m < 12:  level = 'MEDIUM'
    elif score >= 0.20 or depth_m < 25:  level = 'LOW'
    else:                                level = 'SAFE'

    return score, level

print('Risk scorer defined (ego speed + TTC + camera priority).')


Risk scorer defined (ego speed + TTC + camera priority).


In [13]:
# ── CELL 12: Cross-Camera Fusion + World-space NMS ───────────────────────────
#
# THE CORE PROBLEM:
#   When an object (e.g. a car) sits at the boundary between two cameras,
#   YOLO detects it as two separate half-objects. Simple world-space NMS
#   helps, but we can do better:
#
#   1. Edge detection flag: if a bbox touches the image edge, it's likely
#      a partial object that has a matching partial on the neighbor camera
#   2. Neighbor camera matching: pair edge detections from adjacent cameras
#      that project to nearby world locations
#   3. Merge strategy: keep the detection with higher confidence, or if
#      both are edge-clipped, merge their world positions
#
# This approach works WITHOUT calibration data too — in a real car setup,
# you just need to know which cameras are neighbors (left↔front↔right etc.)

# Camera adjacency map (which cameras share a border)
CAM_NEIGHBORS = {
    'CAM_FRONT':       ['CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT'],
    'CAM_FRONT_LEFT':  ['CAM_FRONT', 'CAM_BACK_LEFT'],
    'CAM_FRONT_RIGHT': ['CAM_FRONT', 'CAM_BACK_RIGHT'],
    'CAM_BACK':        ['CAM_BACK_LEFT', 'CAM_BACK_RIGHT'],
    'CAM_BACK_LEFT':   ['CAM_BACK', 'CAM_FRONT_LEFT'],
    'CAM_BACK_RIGHT':  ['CAM_BACK', 'CAM_FRONT_RIGHT'],
}

# Which edge of camera A borders camera B
# 'left' = object at left edge of A might continue in B
# 'right' = object at right edge of A might continue in B
CAM_SHARED_EDGE = {
    ('CAM_FRONT', 'CAM_FRONT_LEFT'):    'left',
    ('CAM_FRONT', 'CAM_FRONT_RIGHT'):   'right',
    ('CAM_FRONT_LEFT', 'CAM_FRONT'):    'right',
    ('CAM_FRONT_RIGHT', 'CAM_FRONT'):   'left',
    ('CAM_FRONT_LEFT', 'CAM_BACK_LEFT'):  'left',
    ('CAM_FRONT_RIGHT', 'CAM_BACK_RIGHT'): 'right',
    ('CAM_BACK', 'CAM_BACK_LEFT'):      'right',
    ('CAM_BACK', 'CAM_BACK_RIGHT'):     'left',
    ('CAM_BACK_LEFT', 'CAM_BACK'):      'left',
    ('CAM_BACK_RIGHT', 'CAM_BACK'):     'right',
    ('CAM_BACK_LEFT', 'CAM_FRONT_LEFT'):  'right',
    ('CAM_BACK_RIGHT', 'CAM_FRONT_RIGHT'): 'left',
}

EDGE_MARGIN = 30  # pixels from image edge to flag as "edge detection"

NMS_THRESHOLDS = {
    'person': 2.0, 'bicycle': 2.0, 'motorcycle': 2.5,
    'car': 3.5, 'truck': 5.0, 'bus': 6.0,
}


def is_edge_detection(bbox, img_w, img_h, margin=EDGE_MARGIN):
    """
    Check if a bbox is clipped at the image edge.
    Returns: 'left', 'right', 'top', 'bottom', or None
    """
    x1, y1, x2, y2 = bbox
    if x1 <= margin:
        return 'left'
    if x2 >= img_w - margin:
        return 'right'
    if y1 <= margin:
        return 'top'
    if y2 >= img_h - margin:
        return 'bottom'
    return None


def cross_camera_fusion(detections):
    """
    Smart fusion of detections from multiple cameras:

    Step 1: Flag edge detections (bbox touching image border)
    Step 2: For edge detections, check if a neighbor camera has a matching
            edge detection at a similar world position → these are the SAME object
    Step 3: Merge pairs (keep higher confidence, average world position)
    Step 4: Standard world-space NMS for remaining duplicates

    Each detection dict must have:
        cname, wx, wy, depth_model, cam, conf, bbox, img_w, img_h
    """
    if not detections:
        return []

    # Step 1: Flag edge detections
    for det in detections:
        det['edge'] = is_edge_detection(
            det.get('bbox', [0,0,0,0]),
            det.get('img_w', 1600),
            det.get('img_h', 900)
        )

    # Step 2: Find cross-camera pairs
    merged_indices = set()
    merged_results = []

    for i, d1 in enumerate(detections):
        if i in merged_indices or d1['edge'] is None:
            continue

        cam1 = d1['cam']
        neighbors = CAM_NEIGHBORS.get(cam1, [])

        best_match = None
        best_dist = float('inf')

        for j, d2 in enumerate(detections):
            if j <= i or j in merged_indices:
                continue
            if d2['cam'] not in neighbors:
                continue
            if d2['edge'] is None:
                continue
            # Check that edges are compatible (d1 right edge ↔ d2 left edge)
            expected_edge_d1 = CAM_SHARED_EDGE.get((cam1, d2['cam']))
            if expected_edge_d1 is None or d1['edge'] != expected_edge_d1:
                continue
            # Same class?
            if d1['cname'] != d2['cname']:
                continue

            # World distance
            dist = math.hypot(d1['wx'] - d2['wx'], d1['wy'] - d2['wy'])
            # Edge pairs from adjacent cameras should be within ~5m
            if dist < 5.0 and dist < best_dist:
                best_dist = dist
                best_match = j

        if best_match is not None:
            d2 = detections[best_match]
            merged_indices.add(i)
            merged_indices.add(best_match)

            # Merge: weighted average position by confidence
            c1 = d1.get('conf', 0.5)
            c2 = d2.get('conf', 0.5)
            total_c = c1 + c2
            merged_det = {
                'cname': d1['cname'],
                'wx': (d1['wx'] * c1 + d2['wx'] * c2) / total_c,
                'wy': (d1['wy'] * c1 + d2['wy'] * c2) / total_c,
                'depth_model': min(d1['depth_model'], d2['depth_model']),
                'cam': d1['cam'] if c1 >= c2 else d2['cam'],
                'conf': max(c1, c2),
                'flow_speed': max(d1.get('flow_speed', 0), d2.get('flow_speed', 0)),
                'merged': True,  # Flag for debugging
                'merge_dist': best_dist,
            }
            merged_results.append(merged_det)

    # Collect unmerged detections
    remaining = [d for i, d in enumerate(detections) if i not in merged_indices]
    all_dets = merged_results + remaining

    # Step 3: Standard world-space NMS on the result
    if not all_dets:
        return []

    dets = sorted(all_dets, key=lambda x: x['depth_model'])
    kept = []
    used = set()

    for i, d1 in enumerate(dets):
        if i in used:
            continue
        kept.append(d1)
        thresh = NMS_THRESHOLDS.get(d1['cname'], 3.0)
        for j, d2 in enumerate(dets):
            if j <= i or j in used:
                continue
            dist = math.hypot(d1['wx'] - d2['wx'], d1['wy'] - d2['wy'])
            if d1['cname'] == d2['cname'] and dist < thresh:
                used.add(j)
            elif dist < 1.5:
                used.add(j)

    return kept

# Backward-compatible alias
nms_world = cross_camera_fusion

print('Cross-camera fusion defined (edge-aware merging + world NMS).')
print(f'  Camera pairs: {len(CAM_SHARED_EDGE)} shared edges')
print(f'  Edge margin: {EDGE_MARGIN}px')


Cross-camera fusion defined (edge-aware merging + world NMS).
  Camera pairs: 12 shared edges
  Edge margin: 30px


In [14]:
# ── CELL 14: BEV renderer (dark theme, speed display, range circles) ────────
MAP_SIZE  = 600
MAP_RANGE = 40.0
SCALE     = MAP_SIZE / (MAP_RANGE * 2)
CENTER    = MAP_SIZE // 2

ANN_SIZE = {
    'vehicle.car':        (4.5, 2.0),
    'vehicle.truck':      (8.0, 2.5),
    'vehicle.bus':       (12.0, 2.8),
    'vehicle.bicycle':    (1.8, 0.8),
    'vehicle.motorcycle': (2.2, 0.9),
    'human.pedestrian':   (0.8, 0.8),
    'movable_object':     (1.0, 1.0),
}
SKIP_CATS = ('movable_object.barrier','movable_object.trafficcone',
             'movable_object.debris','static_object')

def get_ann_size(cat):
    for k,v in ANN_SIZE.items():
        if cat.startswith(k): return v
    return (1.5, 1.5)

def world_to_bev(wx, wy, ego_x, ego_y, ego_yaw):
    dx = wx - ego_x; dy = wy - ego_y
    cos_e = math.cos(-ego_yaw); sin_e = math.sin(-ego_yaw)
    lx =  sin_e*dx + cos_e*dy
    ly =  cos_e*dx - sin_e*dy
    return int(CENTER - lx*SCALE), int(CENTER - ly*SCALE)

def build_map_bg(ego_x, ego_y, ego_yaw):
    canvas = np.full((MAP_SIZE, MAP_SIZE, 3), 40, np.uint8)
    patch  = (ego_x-MAP_RANGE, ego_y-MAP_RANGE,
              ego_x+MAP_RANGE, ego_y+MAP_RANGE)
    for layer in ['road_segment','lane']:
        try: hits = nusc_map.get_records_in_patch(patch,[layer],'intersect')
        except: continue
        for tok in hits.get(layer,[]):
            rec = nusc_map.get(layer, tok)
            pt  = rec.get('polygon_token')
            if not pt: continue
            try: poly = nusc_map.extract_polygon(pt)
            except: continue
            pts = np.array(poly.exterior.coords)
            pxs = np.array([world_to_bev(x,y,ego_x,ego_y,ego_yaw) for x,y in pts], np.int32)
            color = (70, 70, 70) if layer == 'road_segment' else (60, 60, 60)
            cv2.fillPoly(canvas,[pxs],color)
            cv2.polylines(canvas,[pxs],True,(90,90,90),1)
    return canvas

def _draw_ego_and_legend(canvas, ego_speed_ms=0.0):
    ev_l=max(10,int(4.5*SCALE/2)); ev_w=max(6,int(2.0*SCALE/2))
    cv2.rectangle(canvas,(CENTER-ev_w,CENTER-ev_l),(CENTER+ev_w,CENTER+ev_l),(220,120,30),-1)
    cv2.rectangle(canvas,(CENTER-ev_w,CENTER-ev_l),(CENTER+ev_w,CENTER+ev_l),(255,180,60),2)
    tri=np.array([[CENTER,CENTER-ev_l-10],[CENTER-7,CENTER-ev_l+2],[CENTER+7,CENTER-ev_l+2]],np.int32)
    cv2.fillPoly(canvas,[tri],(220,120,30))
    cv2.putText(canvas,'EGO',(CENTER-14,CENTER+5),cv2.FONT_HERSHEY_SIMPLEX,0.38,(255,255,255),1)
    speed_kmh = ego_speed_ms * 3.6
    cv2.putText(canvas, f'{speed_kmh:.0f} km/h', (CENTER-25, CENTER+ev_l+18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200,200,200), 1)
    # Legend
    cv2.rectangle(canvas,(6,6),(200,115),(30,30,30),-1)
    cv2.rectangle(canvas,(6,6),(200,115),(100,100,100),1)
    cv2.putText(canvas,'RISK LEVELS',(12,20),cv2.FONT_HERSHEY_SIMPLEX,0.35,(200,200,200),1)
    for i,(lv,col) in enumerate([('HIGH   (<5m)',RISK_COLOR_BGR['HIGH']),
        ('MEDIUM (5-15m)',RISK_COLOR_BGR['MEDIUM']),
        ('LOW    (15-30m)',RISK_COLOR_BGR['LOW']),
        ('SAFE   (>30m)',RISK_COLOR_BGR['SAFE'])]):
        cy_=35+i*20
        cv2.circle(canvas,(18,cy_),5,col,-1)
        cv2.putText(canvas,lv,(30,cy_+4),cv2.FONT_HERSHEY_SIMPLEX,0.30,(200,200,200),1)
    bar=int(10*SCALE); bx=MAP_SIZE-20-bar; by=MAP_SIZE-20
    cv2.line(canvas,(bx,by),(bx+bar,by),(180,180,180),2)
    cv2.putText(canvas,'10m',(bx,by-5),cv2.FONT_HERSHEY_SIMPLEX,0.38,(180,180,180),1)
    for r in [5,10,20,30]:
        r_px=int(r*SCALE)
        cv2.circle(canvas,(CENTER,CENTER),r_px,(80,80,80),1,cv2.LINE_AA)
        cv2.putText(canvas,f'{r}m',(CENTER+r_px+2,CENTER),
                    cv2.FONT_HERSHEY_SIMPLEX,0.28,(120,120,120),1)

def draw_bev_gt(sample, ego_x, ego_y, ego_yaw, ego_hist, obj_hist, depth_map):
    """BEV from nuScenes Ground Truth annotations."""
    canvas = build_map_bg(ego_x, ego_y, ego_yaw)
    _draw_ego_and_legend(canvas, get_ego_speed(ego_hist))
    if len(ego_hist)>=2:
        pts=[]
        for gx,gy in ego_hist:
            px,py=world_to_bev(gx,gy,ego_x,ego_y,ego_yaw)
            if 0<=px<MAP_SIZE and 0<=py<MAP_SIZE: pts.append((px,py))
        for i in range(1,len(pts)):
            cv2.line(canvas,pts[i-1],pts[i],(0,180,80),2,cv2.LINE_AA)
    for ann_token in sample['anns']:
        ann = nusc.get('sample_annotation', ann_token)
        cat = ann['category_name']
        if any(cat.startswith(s) for s in SKIP_CATS): continue
        pos = ann['translation']
        yaw_a = Quaternion(ann['rotation']).yaw_pitch_roll[0]
        l,w = get_ann_size(cat)
        dx  = pos[0]-ego_x; dy = pos[1]-ego_y
        dist= math.hypot(dx,dy)
        if dist > MAP_RANGE: continue
        score,level = assess_risk(dist, 0.0, cat.split('.')[-1], get_ego_speed(ego_hist))
        color = RISK_COLOR_BGR[level]
        cx_px,cy_px = world_to_bev(pos[0],pos[1],ego_x,ego_y,ego_yaw)
        if not (0<=cx_px<MAP_SIZE and 0<=cy_px<MAP_SIZE): continue
        if ann_token in obj_hist and len(obj_hist[ann_token])>=2:
            th=list(obj_hist[ann_token])
            tpts=[world_to_bev(tx,ty,ego_x,ego_y,ego_yaw) for tx,ty in th]
            tpts=[p for p in tpts if 0<=p[0]<MAP_SIZE and 0<=p[1]<MAP_SIZE]
            for i in range(1,len(tpts)):
                cv2.line(canvas,tpts[i-1],tpts[i],(150,100,200),1)
        if cat.startswith('human.pedestrian'):
            cv2.circle(canvas,(cx_px,cy_px),6,color,-1,cv2.LINE_AA)
            cv2.circle(canvas,(cx_px,cy_px),6,(200,200,200),1,cv2.LINE_AA)
        else:
            bev_angle_deg=-math.degrees(yaw_a-ego_yaw)
            lp=int(l*SCALE); wp=int(w*SCALE)
            box=cv2.boxPoints(((cx_px,cy_px),(wp,lp),bev_angle_deg))
            box=np.intp(box)
            ov=canvas.copy()
            cv2.fillPoly(ov,[box],color)
            cv2.addWeighted(ov,0.5,canvas,0.5,0,canvas)
            cv2.polylines(canvas,[box],True,(200,200,200),1)
            angle_rad=math.radians(bev_angle_deg)
            fx=int(cx_px+(lp/2)*math.sin(angle_rad))
            fy=int(cy_px-(lp/2)*math.cos(angle_rad))
            cv2.arrowedLine(canvas,(cx_px,cy_px),(fx,fy),(255,255,255),2,tipLength=0.4)
        if level in ('HIGH','MEDIUM'):
            cv2.putText(canvas,f'{cat.split(".")[-1][:5]} {dist:.0f}m',
                        (cx_px+int(w*SCALE)+3,cy_px-2),cv2.FONT_HERSHEY_SIMPLEX,0.32,color,1)
    cv2.putText(canvas,'GT',(10,MAP_SIZE-10),cv2.FONT_HERSHEY_SIMPLEX,0.55,(100,220,100),1)
    return canvas

def draw_bev_model(ego_x, ego_y, ego_yaw, ego_hist, all_dets, ego_speed_ms=0.0):
    """BEV from model outputs (YOLO11 + DA2)."""
    canvas = build_map_bg(ego_x, ego_y, ego_yaw)
    _draw_ego_and_legend(canvas, ego_speed_ms)
    if len(ego_hist)>=2:
        pts=[]
        for gx,gy in ego_hist:
            px,py=world_to_bev(gx,gy,ego_x,ego_y,ego_yaw)
            if 0<=px<MAP_SIZE and 0<=py<MAP_SIZE: pts.append((px,py))
        for i in range(1,len(pts)):
            cv2.line(canvas,pts[i-1],pts[i],(0,180,80),2,cv2.LINE_AA)
    for mo in all_dets:
        bx,by=world_to_bev(mo['wx'],mo['wy'],ego_x,ego_y,ego_yaw)
        if not (0<=bx<MAP_SIZE and 0<=by<MAP_SIZE): continue
        flow_spd = mo.get('flow_speed', 0.0)
        score,level=assess_risk(mo['depth_model'], flow_spd, mo['cname'],
                                ego_speed_ms, cam_name=mo.get('cam',''))
        color=RISK_COLOR_BGR[level]
        if mo['cname']=='person':
            cv2.circle(canvas,(bx,by),6,color,-1,cv2.LINE_AA)
            cv2.circle(canvas,(bx,by),6,(200,200,200),1,cv2.LINE_AA)
        else:
            lm,wm={'car':(4.5,2.0),'truck':(8.0,2.5),'bus':(12.0,2.8),
                   'motorcycle':(2.2,0.9),'bicycle':(1.8,0.8)}.get(mo['cname'],(3.0,1.5))
            lp=int(lm*SCALE); wp=int(wm*SCALE)
            bp=cv2.boxPoints(((bx,by),(wp,lp),0)); bp=np.intp(bp)
            ov=canvas.copy()
            cv2.fillPoly(ov,[bp],color)
            cv2.addWeighted(ov,0.5,canvas,0.5,0,canvas)
            cv2.polylines(canvas,[bp],True,(200,200,200),1)
        if level in ('HIGH','MEDIUM'):
            cv2.putText(canvas,f'{mo["cname"][:4]} {mo["depth_model"]:.1f}m',
                        (bx+8,by-2),cv2.FONT_HERSHEY_SIMPLEX,0.32,color,1)
    cv2.putText(canvas,'MODEL',(10,MAP_SIZE-10),cv2.FONT_HERSHEY_SIMPLEX,0.55,(120,120,220),1)
    return canvas

print('BEV renderers defined.')


BEV renderers defined.


In [15]:
# ── CELL 15: Camera frame annotator ─────────────────────────────────────────

def annotate_camera(img_bgr, dets):
    """Draw bounding boxes with risk-level colors on camera image."""
    out = img_bgr.copy()
    for det in dets:
        x1,y1,x2,y2 = [int(v) for v in det['bbox']]
        level  = det.get('level','SAFE')
        cname  = det.get('class_name','?')
        tid    = det.get('track_id',-1)
        depth  = det.get('depth',0.0)
        score  = det.get('score',0.0)
        color  = RISK_COLOR_BGR.get(level,(80,160,80))
        thickness = 3 if level in ('HIGH','MEDIUM') else 2
        cv2.rectangle(out,(x1,y1),(x2,y2),color,thickness)
        tid_str = f'#{tid}' if tid >= 0 else ''
        lines = [f'{tid_str} {cname} [{level}]', f'dist:{depth:.1f}m risk:{score:.0%}']
        for k,line in enumerate(lines):
            tw,th=cv2.getTextSize(line,cv2.FONT_HERSHEY_SIMPLEX,0.44,1)[0]
            ty=y1-4-(len(lines)-1-k)*(th+4)
            if ty<th+4: ty=y2+(k+1)*(th+6)
            cv2.rectangle(out,(x1,ty-th-2),(x1+tw+3,ty+2),color,-1)
            cv2.putText(out,line,(x1+2,ty),
                        cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1,cv2.LINE_AA)
    return out

print('Camera annotator defined.')


def build_unified_strip(sample, cam_list, cam_dets_dict, all_model_dets,
                        strip_w=1920, strip_h=430):
    """
    Build a unified camera strip where 3 cameras look like ONE wide camera.

    The key trick: objects that were merged by cross-camera fusion get drawn
    as a SINGLE bounding box spanning the correct position on the combined image.
    Edge-clipped objects from adjacent cameras are connected into one box.

    Args:
        sample: nuScenes sample dict
        cam_list: ordered camera names [left, center, right]
        cam_dets_dict: per-camera detection lists (from YOLO)
        all_model_dets: world-fused detections (after cross_camera_fusion)
        strip_w, strip_h: output dimensions
    Returns:
        unified strip image (RGB)
    """
    n_cams = len(cam_list)
    cam_w = strip_w // n_cams

    # Step 1: Build raw combined image (no annotations yet)
    raw_strips = []
    cam_imgs = {}
    for cam in cam_list:
        img_c = get_cam_image(sample, cam)
        if img_c is not None:
            resized = cv2.resize(img_c, (cam_w, strip_h))
            raw_strips.append(resized)
            cam_imgs[cam] = (img_c.shape[1], img_c.shape[0])  # original (w, h)
        else:
            raw_strips.append(np.zeros((strip_h, cam_w, 3), np.uint8))
            cam_imgs[cam] = (1600, 900)

    canvas = np.hstack(raw_strips)  # BGR combined

    # Step 2: Camera index mapping (which x-offset for each camera)
    cam_offsets = {}
    for ci, cam in enumerate(cam_list):
        cam_offsets[cam] = ci * cam_w

    # Step 3: Draw non-edge detections normally per camera
    # and collect edge detections for unified drawing
    edge_dets_by_cam = {cam: [] for cam in cam_list}

    for cam in cam_list:
        x_offset = cam_offsets[cam]
        orig_w, orig_h = cam_imgs[cam]
        scale_x = cam_w / orig_w
        scale_y = strip_h / orig_h

        for det in cam_dets_dict.get(cam, []):
            bbox = det['bbox']
            x1_orig, y1_orig, x2_orig, y2_orig = bbox

            # Check if this is an edge detection
            is_left_edge = x1_orig <= EDGE_MARGIN
            is_right_edge = x2_orig >= orig_w - EDGE_MARGIN

            if is_left_edge or is_right_edge:
                edge_dets_by_cam[cam].append(det)
                continue

            # Normal detection: draw on the strip
            x1 = int(x1_orig * scale_x) + x_offset
            y1 = int(y1_orig * scale_y)
            x2 = int(x2_orig * scale_x) + x_offset
            y2 = int(y2_orig * scale_y)

            level = det.get('level', 'SAFE')
            color = RISK_COLOR_BGR.get(level, (80, 160, 80))
            thickness = 3 if level in ('CRITICAL', 'HIGH') else 2
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)

            cname = det.get('class_name', '?')
            depth = det.get('depth', 0.0)
            score = det.get('risk_score', 0.0)
            tid = det.get('track_id', -1)
            tid_s = f'#{tid} ' if tid >= 0 else ''
            label = f'{tid_s}{cname} {depth:.1f}m [{level}]'
            tw, th = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)[0]
            ly = y1 - 6 if y1 > th + 8 else y2 + th + 6
            cv2.rectangle(canvas, (x1, ly - th - 2), (x1 + tw + 4, ly + 2), color, -1)
            cv2.putText(canvas, label, (x1 + 2, ly),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

    # Step 4: Match edge detections across adjacent cameras → draw unified boxes
    used_edge = set()

    for ci in range(n_cams - 1):
        cam_a = cam_list[ci]
        cam_b = cam_list[ci + 1]
        orig_wa, orig_ha = cam_imgs[cam_a]
        orig_wb, orig_hb = cam_imgs[cam_b]
        scale_xa = cam_w / orig_wa
        scale_ya = strip_h / orig_ha
        scale_xb = cam_w / orig_wb
        scale_yb = strip_h / orig_hb
        off_a = cam_offsets[cam_a]
        off_b = cam_offsets[cam_b]

        for di, det_a in enumerate(edge_dets_by_cam[cam_a]):
            key_a = (cam_a, di)
            if key_a in used_edge:
                continue
            ba = det_a['bbox']
            # Only right-edge of cam_a
            if ba[2] < orig_wa - EDGE_MARGIN:
                continue

            best_match = None
            best_iou_y = 0

            for dj, det_b in enumerate(edge_dets_by_cam[cam_b]):
                key_b = (cam_b, dj)
                if key_b in used_edge:
                    continue
                bb = det_b['bbox']
                # Only left-edge of cam_b
                if bb[0] > EDGE_MARGIN:
                    continue
                # Same class?
                if det_a.get('class_name') != det_b.get('class_name'):
                    continue
                # Vertical overlap check (in normalized coords)
                y1_a = ba[1] / orig_ha
                y2_a = ba[3] / orig_ha
                y1_b = bb[1] / orig_hb
                y2_b = bb[3] / orig_hb
                overlap_y = max(0, min(y2_a, y2_b) - max(y1_a, y1_b))
                union_y = max(y2_a, y2_b) - min(y1_a, y1_b)
                iou_y = overlap_y / max(union_y, 1e-6)

                if iou_y > 0.3 and iou_y > best_iou_y:
                    best_iou_y = iou_y
                    best_match = (dj, det_b)

            if best_match is not None:
                dj, det_b = best_match
                used_edge.add(key_a)
                used_edge.add((cam_b, dj))
                bb = det_b['bbox']

                # Unified bbox: from det_a's left to det_b's right
                x1_uni = int(ba[0] * scale_xa) + off_a
                y1_uni = min(int(ba[1] * scale_ya), int(bb[1] * scale_yb))
                x2_uni = int(bb[2] * scale_xb) + off_b
                y2_uni = max(int(ba[3] * scale_ya), int(bb[3] * scale_yb))

                # Use higher risk level
                level_a = det_a.get('level', 'SAFE')
                level_b = det_b.get('level', 'SAFE')
                levels_rank = ['SAFE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
                level = level_a if levels_rank.index(level_a) >= levels_rank.index(level_b) else level_b
                color = RISK_COLOR_BGR.get(level, (80, 160, 80))

                thickness = 3 if level in ('CRITICAL', 'HIGH') else 2
                cv2.rectangle(canvas, (x1_uni, y1_uni), (x2_uni, y2_uni), color, thickness)

                cname = det_a.get('class_name', '?')
                depth = min(det_a.get('depth', 99), det_b.get('depth', 99))
                label = f'{cname} {depth:.1f}m [{level}] (merged)'
                tw, th = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)[0]
                ly = y1_uni - 6 if y1_uni > th + 8 else y2_uni + th + 6
                cv2.rectangle(canvas, (x1_uni, ly - th - 2), (x1_uni + tw + 4, ly + 2), color, -1)
                cv2.putText(canvas, label, (x1_uni + 2, ly),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
            # else: unmatched edge — still draw

    # Step 5: Draw remaining unmatched edge detections normally
    for cam in cam_list:
        x_offset = cam_offsets[cam]
        orig_w, orig_h = cam_imgs[cam]
        scale_x = cam_w / orig_w
        scale_y = strip_h / orig_h

        for di, det in enumerate(edge_dets_by_cam[cam]):
            if (cam, di) in used_edge:
                continue
            bbox = det['bbox']
            x1 = int(bbox[0] * scale_x) + x_offset
            y1 = int(bbox[1] * scale_y)
            x2 = int(bbox[2] * scale_x) + x_offset
            y2 = int(bbox[3] * scale_y)
            level = det.get('level', 'SAFE')
            color = RISK_COLOR_BGR.get(level, (80, 160, 80))
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
            cname = det.get('class_name', '?')
            depth = det.get('depth', 0.0)
            label = f'{cname} {depth:.1f}m'
            tw, th = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.38, 1)[0]
            ly = y1 - 4 if y1 > th + 6 else y2 + th + 6
            cv2.rectangle(canvas, (x1, ly - th - 2), (x1 + tw + 3, ly + 2), color, -1)
            cv2.putText(canvas, label, (x1 + 2, ly),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1, cv2.LINE_AA)

    # Camera boundary lines (subtle)
    for ci in range(1, n_cams):
        bx = ci * cam_w
        cv2.line(canvas, (bx, 0), (bx, strip_h), (100, 100, 100), 1, cv2.LINE_AA)

    # Camera labels
    for ci, cam in enumerate(cam_list):
        cx = cam_offsets[cam] + 8
        cv2.putText(canvas, cam.replace('CAM_', ''), (cx, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 220), 1)

    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

print('Unified strip annotator defined (cross-camera merged bboxes).')


Camera annotator defined.
Unified strip annotator defined (cross-camera merged bboxes).


In [18]:
# ── CELL 16: MAIN PROCESSING LOOP ───────────────────────────────────────────
# === FIX 6: DA2 depth on ALL cameras, StrongSORT on ALL, ego speed in risk ===
MAX_FRAMES   = min(20, len(samples))
FPS_OUT      = 2
VIDEO_OUTPUT = '/content/danger_detection_v4.mp4'
TARGET_H     = 430
TARGET_W     = 640
PANO_W       = TARGET_W * 3
BEV_SQ       = TARGET_H * 3

ego_hist = deque(maxlen=40)
obj_hist = defaultdict(lambda: deque(maxlen=20))
prev_frames = {}  # Reset optical flow state

writer = imageio.get_writer(
    VIDEO_OUTPUT, fps=FPS_OUT, codec='libx264', quality=8, macro_block_size=2)

print(f'Processing {MAX_FRAMES} frames...')

for f_idx in tqdm(range(MAX_FRAMES)):
    sample = samples[f_idx]
    ego_x, ego_y, ego_yaw = get_ego_pose(sample)
    ego_hist.append((ego_x, ego_y))
    ego_speed = get_ego_speed(ego_hist)

    for ann_token in sample['anns']:
        ann = nusc.get('sample_annotation', ann_token)
        obj_hist[ann_token].append(ann['translation'][:2])

    # DA2 depth on ALL 6 cameras
    depth_maps = {}
    for cam in CAM_NAMES:
        img_c = get_cam_image(sample, cam)
        if img_c is not None:
            depth_maps[cam] = run_depth(img_c)

    # Collect detections from all 6 cameras
    all_model_dets = []
    cam_dets = {}

    for cam in CAM_NAMES:
        img_c = get_cam_image(sample, cam)
        if img_c is None: continue
        dm      = depth_maps.get(cam)
        K_c     = get_intrinsics(sample, cam)
        T_c2e_c = get_cam2ego(sample, cam)
        H_c,W_c = img_c.shape[:2]
        dets_raw = []

        res = yolo(img_c, conf=0.20, classes=list(DETECT_CLASSES.keys()), verbose=False)[0]

        if res.boxes is not None and len(res.boxes):
            boxes_np = res.boxes.xyxy.cpu().numpy()
            confs_np = res.boxes.conf.cpu().numpy()
            clss_np  = res.boxes.cls.cpu().numpy()
            dets_in  = np.column_stack([boxes_np, confs_np, clss_np])

            try:
                tracks = trackers[cam].update(dets_in, img_c)
                if tracks is not None and len(tracks):
                    for tr in tracks:
                        x1,y1,x2,y2,tid,conf,cls_id = tr[:7]
                        cid  = int(cls_id)
                        cname = DETECT_CLASSES.get(cid,'object')
                        if cname == 'object': continue
                        d = bbox_depth(dm,[x1,y1,x2,y2]) if dm is not None else float('inf')
                        if d <= 0.3 or d > 60: continue
                        flow = get_flow(cam, img_c)
                        vx, vy, spd, hdg = bbox_flow(flow, [x1,y1,x2,y2])
                        fx_k,fy_k = K_c[0,0], K_c[1,1]
                        px_k,py_k = K_c[0,2], K_c[1,2]
                        cx_img = (x1+x2)/2; cy_img = (y1+y2)/2
                        Xc = (cx_img - px_k) * d / fx_k
                        Yc = (cy_img - py_k) * d / fy_k
                        pt_ego = T_c2e_c @ np.array([Xc, Yc, d, 1.0])
                        c_e, s_e = math.cos(ego_yaw), math.sin(ego_yaw)
                        wx = ego_x + c_e*pt_ego[0] - s_e*pt_ego[1]
                        wy = ego_y + s_e*pt_ego[0] + c_e*pt_ego[1]
                        score, level = assess_risk(d, spd, cname, ego_speed, hdg, cam)
                        dets_raw.append({
                            'bbox':[x1,y1,x2,y2], 'class_name':cname,
                            'track_id':int(tid), 'depth':d, 'level':level,
                            'score':score, 'flow_speed':spd
                        })
                        all_model_dets.append({
                            'cname':cname, 'wx':wx, 'wy':wy,
                            'depth_model':d, 'cam':cam, 'flow_speed':spd, 'score':score,
                            'conf':float(tr[5]) if len(tr)>5 else 0.5,
                            'bbox':[x1,y1,x2,y2], 'img_w':W_c, 'img_h':H_c,
                        })
            except Exception:
                # Fallback without tracker
                for box in res.boxes:
                    x1,y1,x2,y2 = box.xyxy[0].tolist()
                    cid = int(box.cls[0]); cname = DETECT_CLASSES.get(cid,'object')
                    if cname == 'object': continue
                    d = bbox_depth(dm,[x1,y1,x2,y2]) if dm is not None else float('inf')
                    if d <= 0.3 or d > 60: continue
                    fx_k,fy_k = K_c[0,0], K_c[1,1]
                    px_k,py_k = K_c[0,2], K_c[1,2]
                    cx_img = (x1+x2)/2; cy_img = (y1+y2)/2
                    Xc = (cx_img - px_k) * d / fx_k
                    Yc = (cy_img - py_k) * d / fy_k
                    pt_ego = T_c2e_c @ np.array([Xc, Yc, d, 1.0])
                    c_e, s_e = math.cos(ego_yaw), math.sin(ego_yaw)
                    wx = ego_x + c_e*pt_ego[0] - s_e*pt_ego[1]
                    wy = ego_y + s_e*pt_ego[0] + c_e*pt_ego[1]
                    score, level = assess_risk(d, 0.0, cname, ego_speed, 0.0, cam)
                    dets_raw.append({
                        'bbox':[x1,y1,x2,y2], 'class_name':cname,
                        'track_id':-1, 'depth':d, 'level':level,
                        'score':score, 'flow_speed':0.0
                    })
                    all_model_dets.append({
                        'cname':cname, 'wx':wx, 'wy':wy,
                        'depth_model':d, 'cam':cam, 'flow_speed':0.0, 'score':score,
                        'conf':float(box.conf[0]),
                        'bbox':[x1,y1,x2,y2], 'img_w':W_c, 'img_h':H_c,
                    })

        cam_dets[cam] = dets_raw

    # World-space NMS
    all_model_dets = nms_world(all_model_dets)

    # BEV
    bev = draw_bev_model(ego_x, ego_y, ego_yaw, ego_hist, all_model_dets, ego_speed)
    cv2.putText(bev, f'Frame {f_idx+1}/{MAX_FRAMES}  |  {len(all_model_dets)} obj',
                (MAP_SIZE-200,MAP_SIZE-8), cv2.FONT_HERSHEY_SIMPLEX, 0.38,(180,180,180),1)

    # ── Video layout ─────────────────────────────────────────────────────────
    STRIP_W = PANO_W

    # Row 1: Individual front cameras with per-camera annotations
    CAM_W = STRIP_W // 3
    front_strips = []
    for cam in FRONT_CAMS:
        img_c = get_cam_image(sample, cam)
        if img_c is not None:
            ann = annotate_camera(img_c, cam_dets.get(cam, []))
            strip = cv2.resize(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB), (CAM_W, TARGET_H))
            cv2.putText(strip, cam.replace('CAM_',''), (8, 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 220), 1)
        else:
            strip = np.zeros((TARGET_H, CAM_W, 3), np.uint8)
        front_strips.append(strip)
    front_row = np.hstack(front_strips)

    # Row 2: Unified front strip — 3 cameras as ONE wide view
    # Split objects at camera borders are shown as single merged bboxes
    unified_front = build_unified_strip(
        sample, FRONT_CAMS, cam_dets, all_model_dets,
        strip_w=STRIP_W, strip_h=TARGET_H)
    cv2.putText(unified_front, 'UNIFIED FRONT VIEW (merged detections)', (8, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 220), 1)

    # ── BEV canvas ───────────────────────────────────────────────────────────
    bev_rgb    = cv2.cvtColor(bev, cv2.COLOR_BGR2RGB)
    bev_square = cv2.resize(bev_rgb, (BEV_SQ, BEV_SQ))
    bev_canvas = np.zeros((BEV_SQ, STRIP_W, 3), np.uint8)
    x_off = (STRIP_W - BEV_SQ) // 2
    bev_canvas[:, x_off:x_off + BEV_SQ, :] = bev_square

    # ── Combine: Individual cams + Unified view + BEV ────────────────────────
    divider  = np.full((4, STRIP_W, 3), 60, np.uint8)
    combined = np.concatenate([front_row, divider, unified_front, divider, bev_canvas], axis=0)
    h,w = combined.shape[:2]
    if h%2: combined=combined[:h-1]
    if w%2: combined=combined[:,:w-1]
    writer.append_data(combined)

writer.close()
print(f'Video saved: {VIDEO_OUTPUT}')


Processing 20 frames...


100%|██████████| 20/20 [04:41<00:00, 14.09s/it]


Video saved: /content/danger_detection_v4.mp4


In [21]:
# ── CELL 17: All 6 cameras annotated view (single frame) ─────────────────────
sample_view = samples[len(samples) // 2]
ego_x, ego_y, ego_yaw = get_ego_pose(sample_view)
ego_speed_v = get_ego_speed(ego_hist) if len(ego_hist) >= 2 else 0.0

# Run detection on all 6 cameras
view_dets = {}
for cam in CAM_NAMES:
    img_c = get_cam_image(sample_view, cam)
    if img_c is None:
        continue
    dm = run_depth(img_c)
    K_c = get_intrinsics(sample_view, cam)
    T_c2e = get_cam2ego(sample_view, cam)
    T_e2w = get_ego2world(sample_view)
    H_c, W_c = img_c.shape[:2]

    res = yolo(img_c, conf=0.20, classes=list(DETECT_CLASSES.keys()), verbose=False)[0]
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cid = int(box.cls[0])
            cname = DETECT_CLASSES.get(cid, 'object')
            if cname == 'object':
                continue
            d = bbox_depth(dm, [x1, y1, x2, y2])
            if d <= 0.3 or d > 65:
                continue
            score, level = assess_risk(d, 0.0, cname, ego_speed_v)
            dets.append({
                'bbox': [x1, y1, x2, y2], 'class_name': cname,
                'track_id': -1, 'depth': d, 'level': level,
                'score': score, 'risk_score': score,
            })
    view_dets[cam] = dets

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
cam_order = [FRONT_CAMS, BACK_CAMS]
titles = [['FRONT_LEFT', 'FRONT', 'FRONT_RIGHT'],
          ['BACK_RIGHT', 'BACK', 'BACK_LEFT']]

for row in range(2):
    cams = cam_order[row]
    for col in range(3):
        cam = cams[col]
        img_c = get_cam_image(sample_view, cam)
        if img_c is not None:
            ann = annotate_camera(img_c, view_dets.get(cam, []))
            axes[row][col].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        axes[row][col].set_title(f'{titles[row][col]}  ({len(view_dets.get(cam, []))} det)', fontsize=11)
        axes[row][col].axis('off')

plt.suptitle(f'All 6 Cameras — Frame {len(samples)//2}  |  Ego speed: {ego_speed_v*3.6:.0f} km/h', fontsize=13)
plt.tight_layout()
plt.savefig('/content/all_cameras_annotated.png', dpi=120, bbox_inches='tight')
plt.show()
total_dets = sum(len(v) for v in view_dets.values())
print(f'Total detections across 6 cameras: {total_dets}')


Total detections across 6 cameras: 42


In [23]:
# ── CELL 18: BEV comparison — GT vs Model (single frame) ─────────────────────
sample    = samples[10]
ego_x, ego_y, ego_yaw = get_ego_pose(sample)
_ego_hist = deque(maxlen=40)
_ego_hist.append((ego_x, ego_y))

# GT BEV
bev_gt = draw_bev_gt(sample, ego_x, ego_y, ego_yaw,
                     _ego_hist,
                     defaultdict(lambda: deque(maxlen=20)), None)

# Model BEV -- all 6 cameras with DA2
all_dets_dbg = []
for cam in CAM_NAMES:
    img_c = get_cam_image(sample, cam)
    if img_c is None: continue
    dm = run_depth(img_c)
    K_c = get_intrinsics(sample, cam)
    T_c2e_c = get_cam2ego(sample, cam)
    res = yolo(img_c, conf=0.20, classes=list(DETECT_CLASSES.keys()), verbose=False)[0]
    if res.boxes is None: continue
    for box in res.boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        cid = int(box.cls[0]); cname = DETECT_CLASSES.get(cid, 'object')
        if cname == 'object': continue
        d = bbox_depth(dm, [x1,y1,x2,y2])
        if d <= 0.3 or d > 60: continue
        fx_k, fy_k = K_c[0,0], K_c[1,1]
        px_k, py_k = K_c[0,2], K_c[1,2]
        cx_img = (x1+x2)/2; cy_img = (y1+y2)/2
        Xc = (cx_img - px_k) * d / fx_k
        Yc = (cy_img - py_k) * d / fy_k
        pt_ego = T_c2e_c @ np.array([Xc, Yc, d, 1.0])
        c_e, s_e = math.cos(ego_yaw), math.sin(ego_yaw)
        wx = ego_x + c_e*pt_ego[0] - s_e*pt_ego[1]
        wy = ego_y + s_e*pt_ego[0] + c_e*pt_ego[1]
        all_dets_dbg.append({'cname':cname, 'wx':wx, 'wy':wy, 'depth_model':d, 'cam':cam})

all_dets_dbg = nms_world(all_dets_dbg)
bev_model = draw_bev_model(ego_x, ego_y, ego_yaw, _ego_hist, all_dets_dbg, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(cv2.cvtColor(bev_gt, cv2.COLOR_BGR2RGB))
axes[0].set_title('BEV - nuScenes Ground Truth', fontsize=14)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(bev_model, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'BEV - YOLO11 + DA2 ({len(all_dets_dbg)} objects)', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/content/bev_compare.png', dpi=120, bbox_inches='tight')
plt.show()
gt_count = sum(1 for t in sample['anns']
               if not any(nusc.get('sample_annotation',t)['category_name'].startswith(s) for s in SKIP_CATS)
               and math.hypot(nusc.get('sample_annotation',t)['translation'][0]-ego_x,
                              nusc.get('sample_annotation',t)['translation'][1]-ego_y) <= MAP_RANGE)
print(f'GT objects (in range): {gt_count}  |  Model detections: {len(all_dets_dbg)}')


GT objects (in range): 39  |  Model detections: 26


In [27]:
# ── CELL 19: EVALUATION -- GT vs Model (fixed matching + all-camera depth) ──
# === FIX 7: Correct evaluation pipeline ===
#   - DA2 depth on ALL cameras (not just CAM_FRONT!)
#   - Category mapping: nuScenes 'adult' <-> COCO 'person'
#   - Only evaluate GT objects YOLO can actually detect
#   - Category-aware matching
#   - Multiple thresholds for analysis

EVAL_FRAMES = min(len(samples), 20)
MATCH_THRESHOLDS = [3.0, 5.0, 7.0]

results_by_thresh = {}

for DIST_THRESH in MATCH_THRESHOLDS:
    total_gt = 0; total_det = 0; total_tp = 0
    depth_errors = []; missed_cats = {}; fp_count = 0
    tp_by_class = defaultdict(int); gt_by_class = defaultdict(int)

    for f_idx in range(EVAL_FRAMES):
        sample = samples[f_idx]
        ego_x, ego_y, ego_yaw = get_ego_pose(sample)

        # GT objects -- only include categories YOLO can detect
        gt_objects = []
        for ann_token in sample['anns']:
            ann = nusc.get('sample_annotation', ann_token)
            cat = ann['category_name']
            if any(cat.startswith(s) for s in SKIP_CATS): continue
            mapped_coco = NUSCENES_TO_COCO.get(cat)
            if mapped_coco is None: continue  # YOLO can't detect this

            pos = ann['translation']
            dist = math.hypot(pos[0]-ego_x, pos[1]-ego_y)
            if dist > MAP_RANGE: continue

            gt_depth = None
            visible_cam = None
            for cam in CAM_NAMES:
                vis, cam_depth = is_visible_in_cam(sample, cam, pos)
                if vis and cam_depth is not None:
                    gt_depth = cam_depth
                    visible_cam = cam
                    break

            gt_objects.append({
                'cat': cat, 'coco_class': mapped_coco,
                'wx': pos[0], 'wy': pos[1], 'dist': dist,
                'gt_depth': gt_depth, 'visible_cam': visible_cam
            })
            gt_by_class[mapped_coco] += 1
        total_gt += len(gt_objects)

        # Model detections -- ALL 6 cameras with DA2
        model_objects = []
        for cam in CAM_NAMES:
            img_c = get_cam_image(sample, cam)
            if img_c is None: continue
            dm = run_depth(img_c)
            K_c = get_intrinsics(sample, cam)
            T_c2e_c = get_cam2ego(sample, cam)
            res = yolo(img_c, conf=0.20, classes=list(DETECT_CLASSES.keys()), verbose=False)[0]
            if res.boxes is None: continue
            for box in res.boxes:
                x1,y1,x2,y2 = box.xyxy[0].tolist()
                cid = int(box.cls[0])
                cname = DETECT_CLASSES.get(cid, 'object')
                if cname == 'object': continue
                d = bbox_depth(dm, [x1,y1,x2,y2])
                if d <= 0.3 or d > 60: continue
                fx_k, fy_k = K_c[0,0], K_c[1,1]
                px_k, py_k = K_c[0,2], K_c[1,2]
                cx_img = (x1+x2)/2; cy_img = (y1+y2)/2
                Xc = (cx_img - px_k) * d / fx_k
                Yc = (cy_img - py_k) * d / fy_k
                pt_ego = T_c2e_c @ np.array([Xc, Yc, d, 1.0])
                c_e, s_e = math.cos(ego_yaw), math.sin(ego_yaw)
                wx = ego_x + c_e*pt_ego[0] - s_e*pt_ego[1]
                wy = ego_y + s_e*pt_ego[0] + c_e*pt_ego[1]
                model_objects.append({
                    'cname': cname, 'wx': wx, 'wy': wy,
                    'depth_model': d, 'cam': cam,
                    'conf': float(box.conf[0])
                })
        model_objects = nms_world(model_objects)
        total_det += len(model_objects)

        # Category-aware matching
        matched_model = set()
        for gt in gt_objects:
            best_dist = DIST_THRESH; best_idx = -1
            for j, mo in enumerate(model_objects):
                if j in matched_model: continue
                if mo['cname'] != gt['coco_class']: continue
                d_match = math.hypot(gt['wx']-mo['wx'], gt['wy']-mo['wy'])
                if d_match < best_dist:
                    best_dist = d_match; best_idx = j
            if best_idx >= 0:
                total_tp += 1; matched_model.add(best_idx)
                tp_by_class[gt['coco_class']] += 1
                mo = model_objects[best_idx]
                if gt['gt_depth'] is not None and gt['gt_depth'] > 0:
                    err = abs(mo['depth_model'] - gt['gt_depth'])
                    depth_errors.append({
                        'gt': gt['gt_depth'], 'model': mo['depth_model'],
                        'err': err, 'cat': gt['cat']
                    })
            else:
                cat_s = gt['coco_class']
                missed_cats[cat_s] = missed_cats.get(cat_s, 0) + 1
        fp_count += len(model_objects) - len(matched_model)

    precision = total_tp / max(total_det, 1)
    recall = total_tp / max(total_gt, 1)
    f1 = 2*precision*recall / max(precision+recall, 1e-6)
    results_by_thresh[DIST_THRESH] = {
        'precision': precision, 'recall': recall, 'f1': f1,
        'tp': total_tp, 'fp': fp_count, 'fn': total_gt - total_tp,
        'gt': total_gt, 'det': total_det,
        'depth_errors': depth_errors, 'missed': missed_cats,
        'tp_class': dict(tp_by_class), 'gt_class': dict(gt_by_class)
    }

# Print results
print('='*70)
print(' EVALUATION: nuScenes GT  vs  YOLO11 + DA2 (FIXED PIPELINE)')
print('='*70)
for thresh in MATCH_THRESHOLDS:
    r = results_by_thresh[thresh]
    print(f'\n--- Matching threshold: {thresh}m ---')
    print(f'  GT (detectable):   {r["gt"]}')
    print(f'  Model detections:  {r["det"]}')
    print(f'  TP: {r["tp"]}  |  FP: {r["fp"]}  |  FN: {r["fn"]}')
    print(f'  Precision: {r["precision"]:.1%}  |  Recall: {r["recall"]:.1%}  |  F1: {r["f1"]:.1%}')

best_t = max(MATCH_THRESHOLDS, key=lambda t: results_by_thresh[t]['f1'])
r = results_by_thresh[best_t]
print(f'\n{"="*70}')
print(f' BEST @ {best_t}m: P={r["precision"]:.1%}  R={r["recall"]:.1%}  F1={r["f1"]:.1%}')
print(f'{"="*70}')

if r['depth_errors']:
    errs = [e['err'] for e in r['depth_errors']]
    print(f'\n--- Depth Estimation ---')
    print(f'  Samples: {len(errs)}  |  MAE: {np.mean(errs):.2f}m  |  RMSE: {np.sqrt(np.mean([e**2 for e in errs])):.2f}m')
    for lo, hi in [(0,10),(10,25),(25,50)]:
        subset = [e for e in r['depth_errors'] if lo <= e['gt'] < hi]
        if subset:
            print(f'  {lo}-{hi}m: n={len(subset):3d}  MAE={np.mean([e["err"] for e in subset]):.2f}m')

print(f'\n--- Per-class recall @ {best_t}m ---')
for cls in sorted(r['gt_class'].keys()):
    tp = r['tp_class'].get(cls, 0); gt = r['gt_class'][cls]
    print(f'  {cls:15s}  {tp:3d}/{gt:3d}  ({tp/max(gt,1):.0%})')

print(f'\n--- Missed by category ---')
for cat, cnt in sorted(r['missed'].items(), key=lambda x: -x[1])[:8]:
    print(f'  {cat:15s}  missed: {cnt}')


 EVALUATION: nuScenes GT  vs  YOLO11 + DA2 (FIXED PIPELINE)

--- Matching threshold: 3.0m ---
  GT (detectable):   680
  Model detections:  556
  TP: 45  |  FP: 511  |  FN: 635
  Precision: 8.1%  |  Recall: 6.6%  |  F1: 7.3%

--- Matching threshold: 5.0m ---
  GT (detectable):   680
  Model detections:  556
  TP: 85  |  FP: 471  |  FN: 595
  Precision: 15.3%  |  Recall: 12.5%  |  F1: 13.8%

--- Matching threshold: 7.0m ---
  GT (detectable):   680
  Model detections:  556
  TP: 146  |  FP: 410  |  FN: 534
  Precision: 26.3%  |  Recall: 21.5%  |  F1: 23.6%

 BEST @ 7.0m: P=26.3%  R=21.5%  F1=23.6%

--- Depth Estimation ---
  Samples: 145  |  MAE: 3.83m  |  RMSE: 4.32m
  0-10m: n= 21  MAE=3.99m
  10-25m: n= 71  MAE=3.93m
  25-50m: n= 53  MAE=3.62m

--- Per-class recall @ 7.0m ---
  bicycle            0/ 13  (0%)
  car               38/ 84  (45%)
  motorcycle         2/ 14  (14%)
  person            88/518  (17%)
  truck             18/ 51  (35%)

--- Missed by category ---
  person      

In [28]:
# ── CELL 20: Preview + Download ─────────────────────────────────────────────
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

VIDEO_OUTPUT = '/content/danger_detection_v4.mp4'

with open(VIDEO_OUTPUT,'rb') as f:
    data = b64encode(f.read()).decode()

display(HTML(f'''
<div style="text-align:center">
  <h3>Danger Detection System v4</h3>
  <p>YOLO11x + Depth Anything V2 + StrongSORT + RAFT | nuScenes</p>
  <video width="900" controls autoplay loop>
    <source src="data:video/mp4;base64,{data}" type="video/mp4">
  </video>
</div>
'''))

files.download(VIDEO_OUTPUT)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
%cd /content/repo
!cp "/content/drive/MyDrive/Colab Notebooks/danger_detection_v2.ipynb" danger_detection_v2.ipynb
!git add danger_detection_v2.ipynb
!git commit -m "v2: All-camera tracking + cross-camera fusion | F1=23.6% @7m | P=26.3% | R=21.5%"
!git push